# Investigation: Why Does Synthetic Data Degrade BC Policy?

**Goal**: Diagnose why JEPA-augmented BC (Module C3) degrades performance as alpha increases,
and test concrete fixes.

**Hypotheses** (from code review):
1. **Action context bug** — during synthetic generation, the action context is *replaced* with T copies of the same action instead of being *slid* like the embedding context. The predictor was trained on diverse action sequences, so this distribution mismatch degrades rollout quality.
2. **No quality filtering** — all synthetic `(z_hat, a)` pairs are used, even when `cos(z_hat, z_real) < 0.5`. The BC policy learns to map drifted latents to expert actions, hurting generalization.
3. **No normalization between rollout steps** — predictor outputs drift off the DINOv2 manifold as errors compound.
4. **Tiny perturbation** — sigma=0.01 gives negligible diversity (0.2% of embedding norm).

**Investigation plan**:
- Section 1: Load models + data
- Section 2: Quantify synthetic quality (cos_sim vs step, histogram)
- Section 3: Fix the action context bug + regenerate
- Section 4: Add quality filtering
- Section 5: Test sigma=0.0 (no perturbation, isolate rollout quality)
- Section 6: Sanity check — BC on real embeddings (upper bound, no predictor)
- Section 7: Re-run BC with all fixes and compare


## Section 1: Setup & Imports

In [ ]:
import sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch>=2.0', 'matplotlib', 'scikit-learn', 'tqdm', 'numpy', 'pandas', 'scipy', 'pyarrow']:
    pip_install(pkg)

import os, math, random, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Paths (same as module_c)
ROOT = Path(os.getcwd()).resolve()
if not (ROOT / 'data' / 'embeddings').exists():
    if (ROOT.parent / 'data' / 'embeddings').exists():
        ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
EMBED_DIR = DATA_DIR / 'embeddings'
PARQUET_DIR = DATA_DIR / 'robomimic-ph-lift-image' / 'data' / 'chunk-000'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Config (same as module_c)
class Config:
    embed_dim = 384
    action_dim = 7
    d_model = 256
    n_heads = 4
    n_layers_tr = 4
    batch_size = 64
    lr = 1e-4
    weight_decay = 0.05
    warmup_steps = 200
    use_amp = True
    dropout_p = 0.2
    seed = 42
    K_values = [1, 5, 10, 20]
    best_T = 8
    n_train_episodes = 160
    bc_hidden = [256, 128]
    bc_epochs = 30
    bc_batch_size = 64
    N_perturb = 10
    sigma_values = [0.01, 0.05, 0.1]
    rollout_K = 1
    rollout_T = 8
    alpha_values = [0.0, 0.25, 0.5, 0.75, 1.0]

cfg = Config()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = True

print('Setup complete.')


## Section 1b: Load Cached Data & Episode Mapping

In [ ]:
# Load cached embeddings and actions
train_emb = torch.load(EMBED_DIR / 'embeddings_train.pt').float()
val_emb = torch.load(EMBED_DIR / 'embeddings_val.pt').float()
train_act = torch.load(EMBED_DIR / 'actions_train.pt').float()
val_act = torch.load(EMBED_DIR / 'actions_val.pt').float()

print(f'Train embeddings: {train_emb.shape}')
print(f'Val embeddings:   {val_emb.shape}')

# Load all Parquet files for episode boundaries
parquet_files = sorted(PARQUET_DIR.glob('episode_*.parquet'))
print(f'Found {len(parquet_files)} parquet episodes')

all_ep_actions = []
all_ep_n_frames = []

for pf in tqdm(parquet_files, desc='Loading parquets'):
    df = pd.read_parquet(pf)
    actions_raw = np.stack(df['action'].values).astype(np.float32)
    all_ep_actions.append(torch.tensor(actions_raw[::2]))
    all_ep_n_frames.append(len(actions_raw[::2]))

# Verify train/val split
train_frames_total = sum(all_ep_n_frames[:cfg.n_train_episodes])
val_frames_total = sum(all_ep_n_frames[cfg.n_train_episodes:])
assert train_frames_total == train_emb.shape[0], f'Train mismatch: {train_frames_total} vs {train_emb.shape[0]}'
assert val_frames_total == val_emb.shape[0], f'Val mismatch: {val_frames_total} vs {val_emb.shape[0]}'
print(f'Train: {train_frames_total} frames in {cfg.n_train_episodes} episodes')
print(f'Val:   {val_frames_total} frames in {len(parquet_files) - cfg.n_train_episodes} episodes')

# Build episode-to-embedding-index mapping
train_offset = 0
val_offset = 0
ep_to_emb_start = {}

for ep_idx in range(len(parquet_files)):
    n_frames = all_ep_n_frames[ep_idx]
    if ep_idx < cfg.n_train_episodes:
        ep_to_emb_start[ep_idx] = ('train', train_offset)
        train_offset += n_frames
    else:
        ep_to_emb_start[ep_idx] = ('val', val_offset)
        val_offset += n_frames

train_ep_frames = all_ep_n_frames[:cfg.n_train_episodes]
val_ep_frames = all_ep_n_frames[cfg.n_train_episodes:]

print('Data loaded.')


## Section 1c: Model Definitions & Load Checkpoints

We re-define `BCPolicy` and `TransformerPredictor` (identical to Module A/C) and load
the trained checkpoints.


In [ ]:
# --- BCPolicy (same as module_c) ---
class BCPolicy(nn.Module):
    def __init__(self, embed_dim=384, action_dim=7, hidden_dims=[256, 128]):
        super().__init__()
        layers = []
        in_dim = embed_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(in_dim, h), nn.ReLU()])
            in_dim = h
        layers.append(nn.Linear(in_dim, action_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

# --- TransformerPredictor (same as module_a/c) ---
class ActionMLP(nn.Module):
    def __init__(self, action_dim=7, hidden_dim=128, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(action_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, a):
        return self.net(a)

class TransformerPredictor(nn.Module):
    def __init__(self, embed_dim=384, action_dim=7, d_model=256,
                 n_layers=4, n_heads=4, max_seq_len=64, dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.embed_proj = nn.Linear(embed_dim, d_model)
        self.action_mlp = ActionMLP(action_dim, hidden_dim=128, out_dim=d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, batch_first=True,
            dropout=dropout, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predictor = nn.Linear(d_model, embed_dim)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, embeds, actions):
        if embeds.dim() == 2:
            embeds = embeds.unsqueeze(1); actions = actions.unsqueeze(1)
        B, T, _ = embeds.shape
        x = self.embed_proj(embeds) + self.action_mlp(actions)
        x = x + self.pos_embed[:, :T, :]
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        x = self.transformer(x, mask=causal_mask)
        return self.predictor(x[:, -1, :])

# --- Load trained predictor ---
ckpt_path = OUTPUT_DIR / f'module_a_transformer_K{cfg.rollout_K}_T{cfg.rollout_T}.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
rollout_predictor = TransformerPredictor(
    embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
    n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
).to(device)
rollout_predictor.load_state_dict(ckpt['model_state_dict'])
rollout_predictor.eval()
print(f'Loaded predictor K={cfg.rollout_K} T={cfg.rollout_T} (val_cos={ckpt.get("val_cos", "?")})')

# --- Load trained BC baseline (for reference) ---
bc_ckpt = torch.load(OUTPUT_DIR / 'module_c_bc_baseline.pt', map_location=device, weights_only=False)
print(f'BC baseline val MSE: {bc_ckpt["val_mse"]:.6f}')

print('Models loaded.')


## Section 1d: BC Training Utilities (reused from Module C)

In [ ]:
def get_lr(step, warmup_steps, total_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

def cosine_loss(pred, target):
    return 1.0 - F.cosine_similarity(pred, target, dim=-1).mean()

# BC datasets
class BCDataset(Dataset):
    def __init__(self, embeddings, actions, ep_n_frames):
        self.embeddings = embeddings
        self.actions = actions
        self.samples = []
        offset = 0
        for n_frames in ep_n_frames:
            for i in range(n_frames):
                self.samples.append((offset + i, offset + i))
            offset += n_frames
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        emb_idx, act_idx = self.samples[idx]
        return self.embeddings[emb_idx], self.actions[act_idx]

bc_train_ds = BCDataset(train_emb, train_act, train_ep_frames)
bc_val_ds = BCDataset(val_emb, val_act, val_ep_frames)
bc_train_ldr = DataLoader(bc_train_ds, batch_size=cfg.bc_batch_size, shuffle=True,
                         num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)
bc_val_ldr = DataLoader(bc_val_ds, batch_size=cfg.bc_batch_size, shuffle=False,
                        num_workers=0, pin_memory=(device.type=='cuda'))

def bc_train_one_epoch(model, loader, optimizer, scaler, global_step, total_steps):
    model.train()
    epoch_loss = 0.0
    for z, a_target in loader:
        z = z.to(device); a_target = a_target.to(device)
        lr = get_lr(global_step, cfg.warmup_steps, total_steps, cfg.lr)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
        if cfg.use_amp and device.type == 'cuda':
            with autocast():
                a_pred = model(z)
                loss = F.mse_loss(a_pred, a_target)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        else:
            a_pred = model(z)
            loss = F.mse_loss(a_pred, a_target)
            loss.backward()
            optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
        global_step += 1
    return epoch_loss / len(loader), global_step

@torch.no_grad()
def bc_compute_val_mse(model, loader):
    model.eval()
    mse_list = []
    for z, a_target in loader:
        z = z.to(device); a_target = a_target.to(device)
        a_pred = model(z)
        mse = F.mse_loss(a_pred, a_target).item()
        mse_list.append(mse)
    return np.mean(mse_list)

def bc_train_model(model, train_ldr, val_ldr, epochs=None, verbose=True):
    epochs = epochs or cfg.bc_epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=(cfg.use_amp and device.type=='cuda'))
    total_steps = epochs * len(train_ldr)
    gs = 0
    train_losses, val_mses = [], []
    best_mse = float('inf')
    best_state = None
    for epoch in range(1, epochs + 1):
        avg_loss, gs = bc_train_one_epoch(model, train_ldr, optimizer, scaler, gs, total_steps)
        val_mse = bc_compute_val_mse(model, val_ldr)
        train_losses.append(avg_loss)
        val_mses.append(val_mse)
        if val_mse < best_mse:
            best_mse = val_mse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose and (epoch % max(1, epochs // 10) == 0 or epoch == 1 or epoch == epochs):
            print(f'  E {epoch:3d}/{epochs} | loss={avg_loss:.4f} | val_mse={val_mse:.4f} | best={best_mse:.4f}')
    if best_state is not None:
        model.load_state_dict(best_state)
    return {'train_losses': train_losses, 'val_mses': val_mses, 'best_mse': best_mse}

print('BC training utilities ready.')


## Section 2: Diagnose Existing Synthetic Data Quality

Load the **already-generated** synthetic data (sigma=0.01, the best sigma from Module C)
and measure its quality against real embeddings.

Key questions:
1. How fast does `cos(synthetic_z, real_z)` degrade over rollout steps?
2. What fraction of synthetic samples have `cos < 0.8` (likely harmful)?
3. Is the degradation worse for longer episodes?


In [ ]:
# Load existing synthetic data (sigma=0.01 was selected as best in module_c)
syn_data = torch.load(EMBED_DIR / 'synthetic_sigma0.01.pt', weights_only=False)
syn_z_list = syn_data['z']  # list of (N_perturb, ep_len, 384)
syn_a_list = syn_data['a']  # list of (ep_len, 7)

print(f'Number of synthetic episode sets: {len(syn_z_list)}')
print(f'First episode shape: {syn_z_list[0].shape} (N_perturb, ep_len, embed_dim)')

# Compute cos_sim(synthetic, real) for every (episode, perturbation, step)
all_cos_per_step = {}  # step -> list of cos values
all_cos_flat = []       # all cos values regardless of step

for ep_idx, syn_z in enumerate(tqdm(syn_z_list, desc='Computing quality')):
    split_name, start = ep_to_emb_start[ep_idx]
    emb = train_emb if split_name == 'train' else val_emb
    n_frames = syn_z.shape[1]
    real_z = emb[start:start+n_frames]  # (ep_len, 384)
    
    for t in range(n_frames):
        cs = F.cosine_similarity(syn_z[:, t], real_z[t:t+1].expand(syn_z.shape[0], -1), dim=-1)
        if t not in all_cos_per_step:
            all_cos_per_step[t] = []
        all_cos_per_step[t].extend(cs.tolist())
        all_cos_flat.extend(cs.tolist())

print(f'Total synthetic samples: {len(all_cos_flat)}')
print(f'Mean cos_sim: {np.mean(all_cos_flat):.4f}')
print(f'Median cos_sim: {np.median(all_cos_flat):.4f}')
print(f'Fraction cos < 0.8: {np.mean(np.array(all_cos_flat) < 0.8):.4f}')
print(f'Fraction cos < 0.5: {np.mean(np.array(all_cos_flat) < 0.5):.4f}')


In [ ]:
# Plot 1: Mean cos_sim vs rollout step
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1a: Mean cos_sim vs step
steps_sorted = sorted(all_cos_per_step.keys())
mean_cos = [np.mean(all_cos_per_step[s]) for s in steps_sorted]
std_cos = [np.std(all_cos_per_step[s]) for s in steps_sorted]

ax = axes[0]
ax.plot(steps_sorted, mean_cos, 'b-', linewidth=2, label='Mean cos_sim')
ax.fill_between(steps_sorted,
                [m - s for m, s in zip(mean_cos, std_cos)],
                [m + s for m, s in zip(mean_cos, std_cos)],
                alpha=0.2, color='blue')
ax.axhline(y=0.8, color='orange', linestyle='--', label='0.8 threshold')
ax.axhline(y=0.5, color='red', linestyle='--', label='0.5 threshold')
ax.set_xlabel('Rollout step')
ax.set_ylabel('cos_sim(synthetic, real)')
ax.set_title('Synthetic Quality vs Rollout Step\n(sigma=0.01, existing)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 1b: Histogram of all cos_sim values
ax = axes[1]
ax.hist(all_cos_flat, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(x=0.8, color='orange', linestyle='--', linewidth=2, label='0.8 threshold')
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='0.5 threshold')
ax.set_xlabel('cos_sim(synthetic, real)')
ax.set_ylabel('Count')
ax.set_title(f'Distribution of Synthetic Sample Quality\n(n={len(all_cos_flat)})')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 1c: Fraction of samples below threshold vs step
ax = axes[2]
frac_below_08 = [np.mean(np.array(all_cos_per_step[s]) < 0.8) for s in steps_sorted]
frac_below_05 = [np.mean(np.array(all_cos_per_step[s]) < 0.5) for s in steps_sorted]
ax.plot(steps_sorted, frac_below_08, 'o-', label='frac < 0.8', color='orange')
ax.plot(steps_sorted, frac_below_05, 's-', label='frac < 0.5', color='red')
ax.set_xlabel('Rollout step')
ax.set_ylabel('Fraction below threshold')
ax.set_title('Fraction of Bad Samples vs Step')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'investigation_synthetic_quality.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSummary:')
print(f'  Steps where mean cos drops below 0.8: step {next((s for s, m in zip(steps_sorted, mean_cos) if m < 0.8), "never")}')
print(f'  Steps where mean cos drops below 0.5: step {next((s for s, m in zip(steps_sorted, mean_cos) if m < 0.5), "never")}')


## Section 3: Identify the Action Context Bug

The original `generate_synthetic_episode()` in Module C has a bug in how the action
context is updated during rollout:

```python
# BUG (original): replaces ALL T action slots with the current action
a_new = expert_actions[t:t+1].unsqueeze(0).repeat(1, T, 1)
a_ctx = a_new  # -> [a_t, a_t, a_t, ..., a_t]
```

During **training**, the predictor sees T *different* actions `[a_{t-7}, ..., a_t]`.
During **synthetic generation**, it sees T *identical* actions `[a_t, ..., a_t]`.
This is a severe distribution mismatch.

**Fix**: Slide the action context, same as the embedding context:
```python
a_ctx = torch.cat([a_ctx[:, 1:, :], expert_actions[t:t+1].unsqueeze(0)], dim=1)
```

Let's also fix the initial context: instead of repeating z_0 T times, use the real
first T embeddings (warm-start) to give the predictor a proper context.


In [ ]:
# Show the bug with a concrete example
print("=== Action Context Bug Demonstration ===\n")

# Take first episode as example
ep_idx = 0
split_name, start = ep_to_emb_start[ep_idx]
emb = train_emb if split_name == 'train' else val_emb
expert_actions = all_ep_actions[ep_idx]
T = cfg.rollout_T

print(f"Episode {ep_idx}: {len(expert_actions)} frames")
print(f"Expert actions (first {T+3} steps):")
for t in range(min(T+3, len(expert_actions))):
    print(f"  t={t}: a={expert_actions[t][:3].tolist()}... (norm={expert_actions[t].norm():.3f})")

print(f"\n--- What the predictor sees during TRAINING (t={T}) ---")
print(f"Action context: [a_0, a_1, ..., a_{T-1}]  (T different actions)")
print(f"Embed context:  [z_0, z_1, ..., z_{T-1}]  (T different embeddings)")

print(f"\n--- What the predictor sees during SYNTHETIC GENERATION (t={T}, BUGGY) ---")
print(f"Action context: [a_{T}, a_{T}, ..., a_{T}]  (T copies of SAME action)")
print(f"Embed context:  [z_pred_{T-7}, ..., z_pred_{T}]  (T different predictions)")

print(f"\n=> The predictor was NEVER trained on flat action sequences!")
print(f"=> Its predictions under this condition are unreliable.")


## Section 3b: Fixed Synthetic Generation Function

In [ ]:
@torch.no_grad()
def generate_synthetic_episode_fixed(predictor, z_0, real_embeddings, expert_actions,
                                      T, sigma, N_perturb, use_warm_start=True,
                                      normalize_rollout=False):
    """
    Fixed version of generate_synthetic_episode.
    
    Fixes:
    1. Action context is SLID (not replaced) — matches training distribution.
    2. Optional warm-start: use real first T embeddings as initial context (instead of repeating z_0).
    3. Optional L2 normalization of predictions to keep them on the DINOv2 manifold.
    
    Args:
        predictor: JEPA predictor model
        z_0: initial embedding (384,)
        real_embeddings: (ep_len, 384) real embeddings for warm-start and quality measurement
        expert_actions: (ep_len, 7) expert actions
        T: context window
        sigma: perturbation std
        N_perturb: number of perturbations
        use_warm_start: if True, use real first T embeddings as initial context
        normalize_rollout: if True, L2-normalize predictions before feeding back
    Returns:
        synthetic_z: (N_perturb, ep_len, 384)
        cos_per_step: (N_perturb, ep_len) cos_sim with real embeddings (for filtering)
    """
    ep_len = len(expert_actions)
    synthetic_z = torch.zeros(N_perturb, ep_len, cfg.embed_dim)
    cos_per_step = torch.zeros(N_perturb, ep_len)
    
    for n in range(N_perturb):
        # Perturb initial embedding
        z_cur = z_0 + torch.randn(cfg.embed_dim) * sigma
        z_cur = z_cur.to(device)
        
        if use_warm_start and ep_len >= T:
            # Warm-start: use real first T embeddings (with perturbation on the last one)
            z_ctx = real_embeddings[:T].unsqueeze(0).to(device).clone()  # (1, T, 384)
            z_ctx[:, -1, :] = z_cur  # perturb the last one
            a_ctx = expert_actions[:T].unsqueeze(0).to(device)  # (1, T, 7) — real first T actions
            start_t = T
        else:
            # Cold-start: repeat perturbed z_0 T times (original behavior)
            z_ctx = z_cur.unsqueeze(0).unsqueeze(0).repeat(1, T, 1)
            a_ctx = expert_actions[:1].unsqueeze(0).repeat(1, T, 1).to(device)
            start_t = 1
        
        # Store initial embeddings
        for t in range(start_t):
            if t == 0 or (use_warm_start and t < T):
                synthetic_z[n, t] = real_embeddings[t] if use_warm_start else z_cur.cpu()
                cos_per_step[n, t] = 1.0 if use_warm_start else F.cosine_similarity(
                    z_cur.cpu(), real_embeddings[t:t+1], dim=-1).item()
        
        # Autoregressive rollout
        for t in range(start_t, ep_len):
            pred = predictor(z_ctx, a_ctx)  # (1, 384)
            
            # Optional: L2 normalize prediction to stay on manifold
            if normalize_rollout:
                pred = F.normalize(pred, dim=-1) * z_0.norm()
            
            # Store prediction
            synthetic_z[n, t] = pred.squeeze(0).cpu()
            cos_per_step[n, t] = F.cosine_similarity(
                pred.squeeze(0).cpu(), real_embeddings[t:t+1], dim=-1).item()
            
            # Slide BOTH contexts (the fix!)
            z_ctx = torch.cat([z_ctx[:, 1:, :], pred.unsqueeze(1)], dim=1)
            a_ctx = torch.cat([a_ctx[:, 1:, :], expert_actions[t:t+1].unsqueeze(0).to(device)], dim=1)
    
    return synthetic_z, cos_per_step

print('Fixed synthetic generation function ready.')


## Section 3c: Generate Fixed Synthetic Data

Generate synthetic trajectories with the fixed function and compare quality
against the original (buggy) version.

We test 4 variants:
1. **Original (buggy)** — loaded from cache
2. **Fixed (warm-start, sliding actions)** — fix #1 + #2
3. **Fixed + normalization** — fix #1 + #2 + #3
4. **Fixed + sigma=0.0** — no perturbation, isolates rollout quality


In [ ]:
# Generate fixed synthetic data for all 4 variants
variants = {
    'original_bug': None,  # loaded from cache
    'fixed_warmstart': {'use_warm_start': True, 'normalize_rollout': False, 'sigma': 0.01},
    'fixed_warmstart_norm': {'use_warm_start': True, 'normalize_rollout': True, 'sigma': 0.01},
    'fixed_warmstart_sigma0': {'use_warm_start': True, 'normalize_rollout': False, 'sigma': 0.0},
}

variant_syn_data = {}

# Load original from cache
variant_syn_data['original_bug'] = {
    'z_list': syn_z_list,
    'a_list': syn_a_list,
    'cos_list': None,  # compute below
}

# Compute cos for original
print('Computing cos_sim for original (buggy) synthetic data...')
orig_cos_list = []
for ep_idx, syn_z in enumerate(tqdm(syn_z_list, desc='Original cos')):
    split_name, start = ep_to_emb_start[ep_idx]
    emb = train_emb if split_name == 'train' else val_emb
    real_z = emb[start:start+syn_z.shape[1]]
    cos = F.cosine_similarity(syn_z, real_z.unsqueeze(0).expand(syn_z.shape[0], -1, -1), dim=-1)
    orig_cos_list.append(cos)
variant_syn_data['original_bug']['cos_list'] = orig_cos_list

# Generate fixed variants
for vname, vcfg in variants.items():
    if vname == 'original_bug':
        continue
    print(f'\nGenerating variant: {vname}...')
    all_syn_z = []
    all_syn_a = []
    all_cos = []
    
    for ep_idx in tqdm(range(cfg.n_train_episodes), desc=vname):
        split_name, start = ep_to_emb_start[ep_idx]
        emb = train_emb if split_name == 'train' else val_emb
        n_frames = all_ep_n_frames[ep_idx]
        if n_frames < cfg.rollout_T + 1:
            continue
        z_0 = emb[start]
        real_z = emb[start:start+n_frames]
        expert_actions = all_ep_actions[ep_idx]
        
        syn_z, cos_ps = generate_synthetic_episode_fixed(
            rollout_predictor, z_0, real_z, expert_actions,
            T=cfg.rollout_T, sigma=vcfg['sigma'], N_perturb=cfg.N_perturb,
            use_warm_start=vcfg['use_warm_start'],
            normalize_rollout=vcfg['normalize_rollout'],
        )
        all_syn_z.append(syn_z)
        all_syn_a.append(expert_actions)
        all_cos.append(cos_ps)
    
    variant_syn_data[vname] = {
        'z_list': all_syn_z,
        'a_list': all_syn_a,
        'cos_list': all_cos,
    }
    print(f'  Generated {len(all_syn_z)} episodes')

print('\nAll variants generated.')


## Section 3d: Compare Quality Across Variants

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {'original_bug': 'red', 'fixed_warmstart': 'blue',
          'fixed_warmstart_norm': 'green', 'fixed_warmstart_sigma0': 'purple'}

# Plot: Mean cos_sim vs step for each variant
ax = axes[0]
for vname, vdata in variant_syn_data.items():
    cos_per_step = {}
    for ep_idx, cos in enumerate(vdata['cos_list']):
        for t in range(cos.shape[1]):
            if t not in cos_per_step:
                cos_per_step[t] = []
            cos_per_step[t].extend(cos[:, t].tolist())
    
    steps = sorted(cos_per_step.keys())
    means = [np.mean(cos_per_step[s]) for s in steps]
    ax.plot(steps[:30], means[:30], 'o-', linewidth=2, markersize=4,
            color=colors.get(vname, 'gray'), label=vname)

ax.axhline(y=0.8, color='orange', linestyle='--', alpha=0.5, label='0.8 threshold')
ax.set_xlabel('Rollout step')
ax.set_ylabel('Mean cos_sim(synthetic, real)')
ax.set_title('Synthetic Quality: Original (buggy) vs Fixed Variants')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot: Fraction below 0.8 vs step
ax = axes[1]
for vname, vdata in variant_syn_data.items():
    cos_per_step = {}
    for ep_idx, cos in enumerate(vdata['cos_list']):
        for t in range(cos.shape[1]):
            if t not in cos_per_step:
                cos_per_step[t] = []
            cos_per_step[t].extend(cos[:, t].tolist())
    
    steps = sorted(cos_per_step.keys())
    frac_below = [np.mean(np.array(cos_per_step[s]) < 0.8) for s in steps]
    ax.plot(steps[:30], frac_below[:30], 'o-', linewidth=2, markersize=4,
            color=colors.get(vname, 'gray'), label=vname)

ax.set_xlabel('Rollout step')
ax.set_ylabel('Fraction cos < 0.8')
ax.set_title('Bad Sample Fraction: Original vs Fixed')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'investigation_variant_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f'\n{"Variant":<30}{"Mean cos":<12}{"Median cos":<14}{"Frac < 0.8":<12}{"Frac < 0.5":<12}')
print('-' * 80)
for vname, vdata in variant_syn_data.items():
    all_cos = []
    for cos in vdata['cos_list']:
        all_cos.extend(cos.flatten().tolist())
    print(f'{vname:<30}{np.mean(all_cos):<12.4f}{np.median(all_cos):<14.4f}'
          f'{np.mean(np.array(all_cos) < 0.8):<12.4f}{np.mean(np.array(all_cos) < 0.5):<12.4f}')


## Section 4: Quality Filtering

Even with the fix, late rollout steps may have low quality. Let's test whether
**filtering** synthetic samples by a cosine similarity threshold improves BC.

Strategy:
- Only keep synthetic samples where `cos(synthetic_z, real_z) > threshold`
- Test thresholds: 0.0 (no filter), 0.7, 0.8, 0.9
- Use the best variant from Section 3


In [ ]:
# Select the best variant for filtering experiments
# (we'll use fixed_warmstart — the simplest fix without normalization)
best_variant = 'fixed_warmstart'
print(f'Using variant: {best_variant} for filtering experiments')

vdata = variant_syn_data[best_variant]

# Build filtered synthetic datasets at different thresholds
thresholds = [0.0, 0.7, 0.8, 0.9]

filtered_stats = {}
for thresh in thresholds:
    total = 0
    kept = 0
    for ep_idx, (syn_z, cos) in enumerate(zip(vdata['z_list'], vdata['cos_list'])):
        total += cos.numel()
        kept += (cos >= thresh).sum().item()
    filtered_stats[thresh] = {'total': total, 'kept': kept, 'frac_kept': kept / total}
    print(f'  Threshold {thresh:.1f}: kept {kept}/{total} ({kept/total:.4f})')


## Section 5: Sanity Check — BC on Real Embeddings (Upper Bound)

Before re-running with fixed synthetic data, let's establish an **upper bound**:
train BC on the real embeddings rolled out with expert actions (i.e., use the real
next-frame embeddings instead of predictor outputs).

If this works well, it confirms the predictor rollouts are the bottleneck.
If this also fails, the problem is in the mixing strategy or BC architecture.


In [ ]:
# Build "perfect synthetic" data: real embeddings + expert actions
# This is the upper bound — what we'd get with a perfect predictor
perfect_syn_z = []
perfect_syn_a = []

for ep_idx in range(cfg.n_train_episodes):
    split_name, start = ep_to_emb_start[ep_idx]
    emb = train_emb if split_name == 'train' else val_emb
    n_frames = all_ep_n_frames[ep_idx]
    perfect_syn_z.append(emb[start:start+n_frames].unsqueeze(0).repeat(cfg.N_perturb, 1, 1))
    perfect_syn_a.append(all_ep_actions[ep_idx])

print(f'Built perfect synthetic data: {len(perfect_syn_z)} episodes')
print(f'  (These are REAL embeddings — upper bound for synthetic augmentation)')

# Build mixed dataset with perfect synthetic
class MixedBCDatasetFiltered(Dataset):
    """Mix real and synthetic BC samples with optional quality filtering."""
    def __init__(self, real_embeddings, real_actions, ep_n_frames,
                 synthetic_z_list, synthetic_a_list, cos_list=None,
                 alpha=0.5, cos_threshold=0.0):
        self.alpha = alpha
        self.cos_threshold = cos_threshold
        
        # Index real samples
        self.real_samples = []
        offset = 0
        for n_frames in ep_n_frames:
            for i in range(n_frames):
                self.real_samples.append((offset + i, offset + i))
            offset += n_frames
        self.real_emb = real_embeddings
        self.real_act = real_actions
        
        # Flatten synthetic samples with optional filtering
        self.syn_samples = []
        n_filtered = 0
        if cos_list is None:
            cos_list = [None] * len(synthetic_z_list)
        for syn_z, syn_a, cos in zip(synthetic_z_list, synthetic_a_list, cos_list):
            N_perturb, ep_len, _ = syn_z.shape
            for n in range(N_perturb):
                for t in range(ep_len):
                    if cos is not None and cos[n, t].item() < cos_threshold:
                        n_filtered += 1
                        continue
                    self.syn_samples.append((syn_z[n, t], syn_a[t]))
        
        print(f'  Real: {len(self.real_samples)}, Synthetic: {len(self.syn_samples)} '
              f'(filtered out {n_filtered}), alpha={alpha}, thresh={cos_threshold}')
    
    def __len__(self):
        return max(len(self.real_samples), len(self.syn_samples))
    
    def __getitem__(self, idx):
        if torch.rand(1).item() < self.alpha and len(self.syn_samples) > 0:
            si = idx % len(self.syn_samples)
            return self.syn_samples[si]
        else:
            ri = idx % len(self.real_samples)
            emb_i, act_i = self.real_samples[ri]
            return self.real_emb[emb_i], self.real_act[act_i]

print('MixedBCDatasetFiltered ready.')


## Section 6: Re-run BC with Fixes

Test matrix:
1. **Original** (buggy synthetic, no filter) — baseline degradation
2. **Perfect synthetic** (real embeddings) — upper bound
3. **Fixed synthetic** (sliding actions + warm-start), no filter
4. **Fixed + filter 0.8** (only keep high-quality samples)
5. **Fixed + filter 0.9** (only keep best samples)

All at alpha=0.5 (the setting where original showed degradation).


In [ ]:
# Build cos_list for perfect synthetic (all 1.0 since it's real embeddings)
perfect_cos = []
for syn_z in perfect_syn_z:
    perfect_cos.append(torch.ones(syn_z.shape[0], syn_z.shape[1]))

# Test configurations
test_configs = [
    ('original_buggy', variant_syn_data['original_bug']['z_list'],
     variant_syn_data['original_bug']['a_list'],
     variant_syn_data['original_bug']['cos_list'], 0.0),
    ('perfect_synthetic', perfect_syn_z, perfect_syn_a, perfect_cos, 0.0),
    ('fixed_nofilter', vdata['z_list'], vdata['a_list'], vdata['cos_list'], 0.0),
    ('fixed_filter0.8', vdata['z_list'], vdata['a_list'], vdata['cos_list'], 0.8),
    ('fixed_filter0.9', vdata['z_list'], vdata['a_list'], vdata['cos_list'], 0.9),
]

alpha_test = 0.5
results = {}

for name, syn_z, syn_a, cos, thresh in test_configs:
    print(f'\n{"="*60}')
    print(f'Testing: {name} (alpha={alpha_test}, thresh={thresh})')
    print(f'{"="*60}')
    
    mixed_ds = MixedBCDatasetFiltered(
        train_emb, train_act, train_ep_frames,
        syn_z, syn_a, cos, alpha=alpha_test, cos_threshold=thresh
    )
    mixed_ldr = DataLoader(mixed_ds, batch_size=cfg.bc_batch_size, shuffle=True,
                          num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)
    
    model = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                     hidden_dims=cfg.bc_hidden).to(device)
    result = bc_train_model(model, mixed_ldr, bc_val_ldr, epochs=cfg.bc_epochs, verbose=True)
    results[name] = result
    print(f'  -> {name}: best val MSE = {result["best_mse"]:.6f}')

print('\nAll experiments complete.')


## Section 7: Results Comparison

In [ ]:
# Summary table
baseline_mse = bc_ckpt['val_mse']

print(f'\n{"Config":<30}{"Val MSE":<14}{"Δ from baseline":<18}{"Verdict":<15}')
print('-' * 77)
for name, r in results.items():
    delta = r['best_mse'] - baseline_mse
    if delta < -0.001:
        verdict = 'BETTER'
    elif delta < 0.001:
        verdict = 'same'
    else:
        verdict = 'WORSE'
    print(f'{name:<30}{r["best_mse"]:<14.6f}{delta:<+18.6f}{verdict:<15}')

print(f'\n{"Baseline (pure BC)":<30}{baseline_mse:<14.6f}{"—":<18}{"reference":<15}')

# Bar plot
fig, ax = plt.subplots(figsize=(12, 6))
names = list(results.keys())
mses = [results[n]['best_mse'] for n in names]
colors_list = ['red', 'green', 'blue', 'orange', 'purple']
bars = ax.bar(range(len(names)), mses, color=colors_list[:len(names)], alpha=0.7, edgecolor='black')
ax.axhline(y=baseline_mse, color='gray', linestyle='--', linewidth=2, label=f'Baseline (MSE={baseline_mse:.4f})')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_ylabel('Val MSE')
ax.set_title(f'BC Performance: Original vs Fixed vs Perfect (alpha={alpha_test})')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, mse in zip(bars, mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{mse:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'investigation_bc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 8: Full Alpha Sweep with Best Fix

Take the best-performing fix from Section 6 and run the full alpha sweep
{0.0, 0.25, 0.5, 0.75, 1.0} to see if the fix changes the trend.


In [ ]:
# Determine best fix (lowest val MSE)
best_fix_name = min(results.keys(), key=lambda k: results[k]['best_mse'])
print(f'Best fix: {best_fix_name} (MSE={results[best_fix_name]["best_mse"]:.6f})')

# Get the data for the best fix
if best_fix_name == 'perfect_synthetic':
    sweep_syn_z = perfect_syn_z
    sweep_syn_a = perfect_syn_a
    sweep_cos = perfect_cos
    sweep_thresh = 0.0
elif best_fix_name.startswith('fixed'):
    sweep_syn_z = vdata['z_list']
    sweep_syn_a = vdata['a_list']
    sweep_cos = vdata['cos_list']
    sweep_thresh = float(best_fix_name.split('filter')[-1]) if 'filter' in best_fix_name else 0.0
else:
    sweep_syn_z = variant_syn_data['original_bug']['z_list']
    sweep_syn_a = variant_syn_data['original_bug']['a_list']
    sweep_cos = variant_syn_data['original_bug']['cos_list']
    sweep_thresh = 0.0

print(f'Using threshold={sweep_thresh} for sweep')

# Run alpha sweep
sweep_results = {}
for alpha in cfg.alpha_values:
    print(f'\n--- alpha={alpha} ---')
    mixed_ds = MixedBCDatasetFiltered(
        train_emb, train_act, train_ep_frames,
        sweep_syn_z, sweep_syn_a, sweep_cos,
        alpha=alpha, cos_threshold=sweep_thresh
    )
    mixed_ldr = DataLoader(mixed_ds, batch_size=cfg.bc_batch_size, shuffle=True,
                          num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)
    
    model = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                     hidden_dims=cfg.bc_hidden).to(device)
    result = bc_train_model(model, mixed_ldr, bc_val_ldr, epochs=cfg.bc_epochs, verbose=False)
    sweep_results[alpha] = result
    print(f'  alpha={alpha}: val MSE = {result["best_mse"]:.6f}')

print('\nAlpha sweep complete.')


In [ ]:
# Plot alpha sweep: original vs fixed
fig, ax = plt.subplots(figsize=(10, 6))

# Original (from module_c, loaded earlier)
orig_alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
orig_mses = []
for a in orig_alphas:
    ckpt = torch.load(OUTPUT_DIR / f'module_c_bc_aug_alpha{a}.pt', map_location='cpu', weights_only=False)
    orig_mses.append(ckpt['val_mse'])

ax.plot(orig_alphas, orig_mses, 'o-', linewidth=2, markersize=8, color='red', label='Original (buggy)')

# Fixed
fixed_alphas = sorted(sweep_results.keys())
fixed_mses = [sweep_results[a]['best_mse'] for a in fixed_alphas]
ax.plot(fixed_alphas, fixed_mses, 's-', linewidth=2, markersize=8, color='blue', label=f'Fixed ({best_fix_name})')

ax.axhline(y=baseline_mse, color='gray', linestyle='--', linewidth=1.5, label=f'Baseline (MSE={baseline_mse:.4f})')
ax.set_xlabel('alpha (synthetic mix ratio)')
ax.set_ylabel('Val MSE')
ax.set_title(f'Alpha Sweep: Original vs Fixed (thresh={sweep_thresh})')
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate values
for a, m in zip(orig_alphas, orig_mses):
    ax.annotate(f'{m:.4f}', (a, m), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9, color='red')
for a, m in zip(fixed_alphas, fixed_mses):
    ax.annotate(f'{m:.4f}', (a, m), textcoords='offset points', xytext=(0, -15), ha='center', fontsize=9, color='blue')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'investigation_alpha_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print(f'\n{"alpha":<8}{"Original MSE":<16}{"Fixed MSE":<16}{"Δ":<12}')
print('-' * 52)
for a in orig_alphas:
    o = orig_mses[orig_alphas.index(a)]
    f = sweep_results.get(a, {}).get('best_mse', float('nan'))
    print(f'{a:<8}{o:<16.6f}{f:<16.6f}{f-o:<+12.6f}')


## Section 9: Conclusions & Next Steps

Fill in based on results above:

### Findings
1. **Action context bug**: (fill in — did the fix help?)
2. **Quality filtering**: (fill in — which threshold worked best?)
3. **Sigma=0.0**: (fill in — was perturbation the issue?)
4. **Perfect synthetic upper bound**: (fill in — how much better is perfect data?)

### Recommendations
- (fill in based on results)
- If the fix helps: update `module_c_bc_repa.ipynb` with the fixed generation function
- If filtering helps: add filtering to the MixedBCDataset
- If perfect synthetic works but fixed doesn't: the predictor quality is the bottleneck — consider training a better predictor (multi-step loss, SIGReg, etc.)

### Save Results


In [ ]:
# Save all investigation results
investigation_results = {
    'synthetic_quality': {
        'original_mean_cos': np.mean(all_cos_flat),
        'original_median_cos': np.median(all_cos_flat),
        'original_frac_below_08': np.mean(np.array(all_cos_flat) < 0.8),
        'original_frac_below_05': np.mean(np.array(all_cos_flat) < 0.5),
    },
    'variant_comparison': {
        name: {
            'best_mse': results[name]['best_mse'],
        } for name in results.keys()
    },
    'alpha_sweep': {
        a: sweep_results[a]['best_mse'] for a in sweep_results.keys()
    },
    'baseline_mse': baseline_mse,
    'best_fix': best_fix_name,
}

torch.save(investigation_results, OUTPUT_DIR / 'investigation_bc_results.pt')
print('Saved investigation results to outputs/investigation_bc_results.pt')
print('\nInvestigation complete!')
